# LeetCode 2116 — Check if a Parentheses String Can Be Valid bằng Python

Link đề: https://leetcode.com/problems/check-if-a-parentheses-string-can-be-valid/ · Độ khó: **Medium** · Chủ đề: String, Stack, Greedy

Skeleton LeetCode:

```python
class Solution:
    def canBeValid(self, s: str, locked: str) -> bool:
        ...
```

## 1. Mục tiêu học

1. Nắm chắc tiêu chí "chuỗi ngoặc hợp lệ" qua **số dư (balance)**: mọi tiền tố có balance ≥ 0 và tổng balance = 0.
2. Hiểu kỹ thuật **khoảng balance khả dĩ `[lo, hi]`** khi có ký tự "joker" (đổi được), và vai trò của **tính chẵn lẻ**.
3. Cài đặt lời giải **quét hai chiều** (trái→phải và phải→trái) và chứng minh vì sao cần cả hai chiều.
4. Cài đặt lời giải **hai ngăn xếp chỉ số** (ngoặc mở bị khóa / vị trí tự do) — và dùng nó để **dựng ra** một chuỗi hợp lệ cụ thể.
5. Đối chiếu tất cả với **vét cạn 2^k** trên input nhỏ để tin tưởng tuyệt đối vào lời giải tham lam.

## 2. Đề bài & ví dụ

Chuỗi ngoặc là chuỗi khác rỗng chỉ gồm `(` và `)`. Nó **hợp lệ** nếu là `()`, hoặc ghép `AB` của hai chuỗi hợp lệ, hoặc `(A)` với `A` hợp lệ.

Cho chuỗi ngoặc `s` và chuỗi nhị phân `locked` cùng độ dài `n`:
- `locked[i] == '1'`: **không được đổi** `s[i]`.
- `locked[i] == '0'`: được đổi `s[i]` thành `(` hoặc `)` tùy ý (giá trị ban đầu của `s[i]` **không quan trọng**).

Trả về `True` nếu có thể làm `s` hợp lệ.

**Ví dụ 1:** `s = "))()))"`, `locked = "010100"` → `True` (đổi `s[0]`, `s[4]` thành `(`: `"()()()"`).

**Ví dụ 2:** `s = "()()"`, `locked = "0000"` → `True`.

**Ví dụ 3:** `s = ")"`, `locked = "0"` → `False` (độ dài lẻ thì không bao giờ hợp lệ).

**Ví dụ 4:** `s = "(((())(((())"`, `locked = "111111010111"` → `True` (đổi `s[6]`, `s[8]` thành `)`).

**Ràng buộc:** `n == s.length == locked.length`, `1 ≤ n ≤ 10⁵`, `s[i] ∈ {'(', ')'}`, `locked[i] ∈ {'0','1'}`.

In [1]:
import random, itertools, time
from typing import List

examples = [
    ("))()))", "010100", True),
    ("()()", "0000", True),
    (")", "0", False),
    ("(((())(((())", "111111010111", True),
]
for s, lk, exp in examples:
    view = "".join(c if k == "1" else "?" for c, k in zip(s, lk))
    print(f"s={s:<14} locked={lk:<14} -> nhìn như '{view}', kỳ vọng {exp}")

s=))()))         locked=010100         -> nhìn như '?)?)??', kỳ vọng True
s=()()           locked=0000           -> nhìn như '????', kỳ vọng True
s=)              locked=0              -> nhìn như '?', kỳ vọng False
s=(((())(((())   locked=111111010111   -> nhìn như '(((())?(?())', kỳ vọng True


## 3. Kiến thức nền

### 3.1 Kiểm tra chuỗi ngoặc hợp lệ bằng balance

Đi từ trái sang phải, `(` cộng 1, `)` trừ 1. Chuỗi hợp lệ ⇔ **không lúc nào âm** và **kết thúc bằng 0**. Đây là định nghĩa tương đương với định nghĩa đệ quy của đề (có thể chứng minh bằng quy nạp: tiền tố đầu tiên quay về 0 chính là khối `(A)`).

In [2]:
def is_valid(t):
    bal = 0
    for c in t:
        bal += 1 if c == "(" else -1
        if bal < 0:
            return False
    return bal == 0

for t in ["()", "(())()", ")(", "(()", "())(()"]:
    print(f"{t:<8} -> {is_valid(t)}")

()       -> True
(())()   -> True
)(       -> False
(()      -> False
())(()   -> False


### 3.2 Duyệt song song với `zip` và duyệt ngược với `reversed`

`zip(s, locked)` cho từng cặp `(ký tự, khóa)`. `reversed(range(n))` hoặc `zip(reversed(s), reversed(locked))` cho thứ tự từ phải sang trái mà không cần tạo chuỗi mới.

In [3]:
s, lk = "(()", "100"
print(list(zip(s, lk)))
print(list(zip(reversed(s), reversed(lk))))
for i in reversed(range(len(s))):
    print(i, s[i], lk[i], end=" | ")
print()

[('(', '1'), ('(', '0'), (')', '0')]
[(')', '0'), ('(', '0'), ('(', '1')]
2 ) 0 | 1 ( 0 | 0 ( 1 | 


### 3.3 `itertools.product` — sinh mọi tổ hợp cho vét cạn

`product("()", repeat=k)` sinh `2^k` cách gán cho `k` vị trí tự do. Chỉ dùng được khi `k` nhỏ (k ≤ 16 là ~65 nghìn cách).

In [4]:
print(list(itertools.product("()", repeat=2)))
print(sum(1 for _ in itertools.product("()", repeat=10)), "cách với k = 10")

[('(', '('), ('(', ')'), (')', '('), (')', ')')]
1024 cách với k = 10


### 3.4 Tính chẵn lẻ của balance

Mỗi ký tự thay đổi balance đúng ±1 ⇒ sau `i` ký tự, balance luôn **cùng tính chẵn lẻ với `i`**. Hệ quả:
- `n` lẻ ⇒ balance cuối lẻ ⇒ không thể bằng 0 ⇒ **luôn False**.
- Tập balance có thể đạt sau `i` bước là các số cùng chẵn lẻ với `i` trong một đoạn — đây là chìa khóa của cách `[lo, hi]`.

In [5]:
t = "(()))("
bal = 0
for i, c in enumerate(t, 1):
    bal += 1 if c == "(" else -1
    print(f"sau {i} ký tự: balance={bal:>2}, i%2={i%2}, bal%2={bal%2}")

sau 1 ký tự: balance= 1, i%2=1, bal%2=1
sau 2 ký tự: balance= 2, i%2=0, bal%2=0
sau 3 ký tự: balance= 1, i%2=1, bal%2=1
sau 4 ký tự: balance= 0, i%2=0, bal%2=0
sau 5 ký tự: balance=-1, i%2=1, bal%2=1
sau 6 ký tự: balance= 0, i%2=0, bal%2=0


## 4. Ý tưởng nền tảng / trực giác

Coi vị trí `locked[i] = '0'` là dấu `?` (joker). Ví dụ 1 trở thành `? ) ? ) ? ?`.

Tính tay "balance có thể là bao nhiêu" sau từng ký tự, **bỏ các giá trị âm** (vì âm là chết ngay):

| i | ký tự | balance có thể |
|---|---|---|
| 0 | — | {0} |
| 1 | ? | {1} (−1 bị loại) |
| 2 | ) | {0} |
| 3 | ? | {1} |
| 4 | ) | {0} |
| 5 | ? | {1} |
| 6 | ? | {0, 2} |

Cuối cùng 0 nằm trong tập ⇒ `True`.

Quan sát:
1. Tập balance khả dĩ luôn có dạng **mọi số cùng chẵn lẻ trong một đoạn `[lo, hi]`** ⇒ chỉ cần lưu 2 số.
2. `?` mở rộng đoạn ra hai phía (`lo − 1`, `hi + 1`); `(` khóa dịch cả đoạn lên; `)` khóa dịch xuống.
3. Nếu `hi < 0` thì ngay cả khi biến mọi `?` thành `(` vẫn âm ⇒ chết.
4. Cuối cùng cần `0 ∈ tập` ⇔ `lo == 0` (sau khi kẹp `lo ≥ 0`) và `n` chẵn.

In [6]:
def possible_balances(s, lk):
    cur = {0}
    rows = []
    for c, k in zip(s, lk):
        nxt = set()
        for b in cur:
            options = ["(", ")"] if k == "0" else [c]
            for o in options:
                nb = b + (1 if o == "(" else -1)
                if nb >= 0:
                    nxt.add(nb)
        cur = nxt
        rows.append(("?" if k == "0" else c, sorted(cur)))
    return rows

for i, (ch, bs) in enumerate(possible_balances("))()))", "010100"), 1):
    print(f"i={i} '{ch}' -> {bs}")

i=1 '?' -> [1]
i=2 ')' -> [0]
i=3 '?' -> [1]
i=4 ')' -> [0]
i=5 '?' -> [1]
i=6 '?' -> [0, 2]


## 5. Cách 1 — Brute Force / baseline: thử mọi cách gán cho vị trí tự do

**Ý tưởng:** gọi `k` = số vị trí `locked = '0'`. Thử cả `2^k` cách gán, cách nào cho chuỗi hợp lệ thì trả `True`.

In [7]:
def brute(s, locked):
    free = [i for i, k in enumerate(locked) if k == "0"]
    base = list(s)
    for choice in itertools.product("()", repeat=len(free)):
        for i, c in zip(free, choice):
            base[i] = c
        if is_valid(base):
            return True
    return False

for s, lk, exp in examples:
    print(s, lk, brute(s, lk), "OK" if brute(s, lk) == exp else "SAI")

))())) 010100 True OK
()() 0000 True OK
) 0 False OK
(((())(((()) 111111010111 True OK


**Độ phức tạp:** O(2^k · n). Với `n = 10⁵` và `k` lớn thì không tưởng: `k = 40` đã là 10¹² cách.

**Nút thắt:** ta thử lại rất nhiều cách gán chỉ khác nhau ở những chi tiết "không ảnh hưởng" — điều quan trọng sau mỗi tiền tố **chỉ là balance**, không phải cách gán cụ thể. Hai cách gán khác nhau cho cùng balance ở bước `i` thì tương lai của chúng như nhau.

### 5.1 Baseline tốt hơn: DP trên tập balance — O(n²)

Từ nhận xét trên: lưu **tập balance khả dĩ** (như mục 4). Kích thước tập ≤ n ⇒ O(n²). Đúng nhưng vẫn chậm với n = 10⁵ (10¹⁰ phép).

In [8]:
def dp_set(s, locked):
    if len(s) % 2:
        return False
    cur = {0}
    for c, k in zip(s, locked):
        nxt = set()
        for b in cur:
            if k == "0" or c == "(":
                nxt.add(b + 1)
            if (k == "0" or c == ")") and b > 0:
                nxt.add(b - 1)
        cur = nxt
        if not cur:
            return False
    return 0 in cur

for s, lk, exp in examples:
    print(s, lk, dp_set(s, lk), "OK" if dp_set(s, lk) == exp else "SAI")

))())) 010100 True OK
()() 0000 True OK
) 0 False OK
(((())(((()) 111111010111 True OK


In [9]:
# Kích thước tập balance có thể tăng tuyến tính -> O(n^2)
for n in [200, 2000]:
    s = "(" * n; lk = "0" * n
    t = time.perf_counter(); dp_set(s, lk); dt = time.perf_counter() - t
    print(f"n={n}: {dt*1000:.1f} ms")

n=200: 0.7 ms
n=2000: 91.5 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ cần hai đầu mút.** Tập balance khả dĩ (đã bỏ số âm) là "mọi số cùng chẵn lẻ trong `[lo, hi]`". Cập nhật:
- `?`: `lo -= 1`, `hi += 1`.
- `(` khóa: `lo += 1`, `hi += 1`.
- `)` khóa: `lo -= 1`, `hi -= 1`.
- Sau đó: **nếu** `hi < 0` **thì** trả `False`; **nếu** `lo < 0` **thì** `lo = max(lo, 0)` (bỏ phần âm).
- Cuối: `n` chẵn **và** `lo == 0`.

Vì sao `max(lo, 0)` đủ dù làm lệch chẵn lẻ? `lo` chỉ âm khi bằng −1 (vì trước đó `lo ≥ 0`), giá trị thật nhỏ nhất là 1, ta ghi 0. Như vậy `lo` ghi lại luôn ∈ {lo_thật − 1, lo_thật}. Ở cuối, `n` chẵn ⇒ lo_thật chẵn; `lo == 0` ⇔ lo_thật ∈ {0, 1} ⇔ lo_thật = 0. (Phiên bản "chuẩn chẵn lẻ" là `lo = 1` khi `lo = −1`, ta cũng sẽ cài.)

**Insight 2 — quét hai chiều.** Tương đương nhưng dễ nhớ hơn:
- **Trái→phải:** ở mọi tiền tố, số vị trí *có thể làm `(`* (= `(` khóa + tự do) phải ≥ số `)` khóa. Nếu không, có một `)` khóa không thể được ghép.
- **Phải→trái:** ở mọi hậu tố, số vị trí *có thể làm `)`* (= `)` khóa + tự do) phải ≥ số `(` khóa.
- Cộng điều kiện `n` chẵn. Ba điều kiện này là **cần và đủ**.

**Insight 3 — ghép bằng ngăn xếp.** Mỗi `)` khóa nên ghép với `(` khóa gần nhất bên trái (nếu có), nếu không thì dùng một vị trí tự do bên trái. Mỗi `(` khóa còn dư phải ghép với một vị trí tự do **bên phải** nó. Phần tự do còn lại tự ghép với nhau (số lượng chẵn khi `n` chẵn).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Khoảng balance `[lo, hi]`

Đoạn code: khởi tạo `lo = hi = 0`; với mỗi ký tự cập nhật theo bảng ở mục 6; nếu `hi < 0` trả `False`; kẹp `lo`. Ta viết bản **chuẩn chẵn lẻ** (khi `lo = −1` thì đặt `lo = 1`), nhờ đó không cần lập luận tinh tế ở cuối, nhưng vẫn kiểm tra `n` chẵn cho chắc.

In [10]:
def range_lo_hi(s, locked):
    if len(s) % 2:
        return False
    lo = hi = 0
    for c, k in zip(s, locked):
        if k == "0":
            lo, hi = lo - 1, hi + 1
        elif c == "(":
            lo, hi = lo + 1, hi + 1
        else:
            lo, hi = lo - 1, hi - 1
        if hi < 0:
            return False
        if lo < 0:
            lo += 2          # -1 -> 1: giữ đúng tính chẵn lẻ
    return lo == 0

for s, lk, exp in examples:
    print(s, lk, range_lo_hi(s, lk), "OK" if range_lo_hi(s, lk) == exp else "SAI")

))())) 010100 True OK
()() 0000 True OK
) 0 False OK
(((())(((()) 111111010111 True OK


Code nộp LeetCode cho cách 2 (bản `max(lo, 0)` ngắn gọn — đúng nhờ lập luận chẵn lẻ ở mục 6):

In [11]:
class SolutionRange:
    def canBeValid(self, s: str, locked: str) -> bool:
        if len(s) % 2:
            return False
        lo = hi = 0
        for c, k in zip(s, locked):
            if k == "0":
                lo -= 1; hi += 1
            elif c == "(":
                lo += 1; hi += 1
            else:
                lo -= 1; hi -= 1
            if hi < 0:
                return False
            lo = max(lo, 0)
        return lo == 0

print([SolutionRange().canBeValid(s, lk) for s, lk, _ in examples])

[True, True, False, True]


**Độ phức tạp cách 2:** O(n) thời gian, O(1) bộ nhớ.

### 7.2 Cách 3 — Quét hai chiều (greedy đếm)

- Trái→phải: `x` = số vị trí có thể làm `(` trừ số `)` khóa. Cụ thể: tự do hoặc `(` ⇒ `x += 1`; `)` khóa ⇒ `x -= 1`. Nếu `x < 0` ⇒ `False`.
- Phải→trái: đối xứng: tự do hoặc `)` ⇒ `x += 1`; `(` khóa ⇒ `x -= 1`.

Chú ý: ở quét trái→phải, `)` **tự do** được tính là "có thể làm `(`", vì giá trị ban đầu của ký tự tự do không quan trọng.

In [12]:
def two_pass(s, locked):
    n = len(s)
    if n % 2:
        return False
    x = 0
    for i in range(n):
        if locked[i] == "0" or s[i] == "(":
            x += 1
        else:
            x -= 1
            if x < 0:
                return False
    x = 0
    for i in range(n - 1, -1, -1):
        if locked[i] == "0" or s[i] == ")":
            x += 1
        else:
            x -= 1
            if x < 0:
                return False
    return True

for s, lk, exp in examples:
    print(s, lk, two_pass(s, lk), "OK" if two_pass(s, lk) == exp else "SAI")

))())) 010100 True OK
()() 0000 True OK
) 0 False OK
(((())(((()) 111111010111 True OK


In [13]:
class SolutionTwoPass:
    def canBeValid(self, s: str, locked: str) -> bool:
        n = len(s)
        if n & 1:
            return False
        x = 0
        for c, k in zip(s, locked):
            x += 1 if (k == "0" or c == "(") else -1
            if x < 0:
                return False
        x = 0
        for c, k in zip(reversed(s), reversed(locked)):
            x += 1 if (k == "0" or c == ")") else -1
            if x < 0:
                return False
        return True

print([SolutionTwoPass().canBeValid(s, lk) for s, lk, _ in examples])

[True, True, False, True]


**Độ phức tạp cách 3:** O(n) thời gian, O(1) bộ nhớ (hai vòng lặp).

### 7.3 Cách 4 — Hai ngăn xếp chỉ số (và dựng chuỗi hợp lệ)

- `opens`: chỉ số các `(` **khóa** chưa được ghép.
- `free`: chỉ số các vị trí **tự do** chưa dùng.

Duyệt trái→phải:
- `(` khóa ⇒ push vào `opens`.
- tự do ⇒ push vào `free`.
- `)` khóa ⇒ ưu tiên pop `opens` (ghép với `(` khóa gần nhất); nếu rỗng thì pop `free` (biến vị trí đó thành `(`); nếu cả hai rỗng ⇒ `False`.

Kết thúc: mỗi `(` khóa còn trong `opens` cần một vị trí tự do **có chỉ số lớn hơn** để làm `)`. Ghép từ đỉnh hai ngăn xếp: nếu `free[-1] > opens[-1]` thì pop cả hai, ngược lại ⇒ `False`. Cuối cùng `opens` rỗng và số `free` còn lại phải chẵn.

**Vì sao ưu tiên `opens` trước `free`?** Vị trí tự do linh hoạt hơn (dùng được cho cả hai vai), nên để dành. Dùng `(` khóa gần nhất (đỉnh ngăn xếp) để những `(` khóa xa hơn (bên trái) lại có nhiều vị trí tự do bên phải hơn.

In [14]:
def stacks(s, locked):
    opens, free = [], []
    for i, (c, k) in enumerate(zip(s, locked)):
        if k == "0":
            free.append(i)
        elif c == "(":
            opens.append(i)
        elif opens:
            opens.pop()
        elif free:
            free.pop()
        else:
            return False
    while opens and free and free[-1] > opens[-1]:
        opens.pop(); free.pop()
    return not opens and len(free) % 2 == 0

for s, lk, exp in examples:
    print(s, lk, stacks(s, lk), "OK" if stacks(s, lk) == exp else "SAI")

))())) 010100 True OK
()() 0000 True OK
) 0 False OK
(((())(((()) 111111010111 True OK


Mở rộng cách 4 để **dựng** một chuỗi hợp lệ (rất hữu ích để kiểm chứng: ta có thể `is_valid` kết quả và kiểm tra các vị trí khóa không bị đổi):

In [15]:
def build_valid(s, locked):
    t = list(s)
    opens, free = [], []
    for i, (c, k) in enumerate(zip(s, locked)):
        if k == "0":
            free.append(i)
        elif c == "(":
            opens.append(i)
        elif opens:
            opens.pop()
        elif free:
            t[free.pop()] = "("
        else:
            return None
    while opens and free and free[-1] > opens[-1]:
        opens.pop(); t[free.pop()] = ")"
    if opens or len(free) % 2:
        return None
    half = len(free) // 2
    for j, i in enumerate(free):          # free tăng dần: nửa đầu '(' nửa sau ')'
        t[i] = "(" if j < half else ")"
    return "".join(t)

for s, lk, _ in examples:
    print(s, lk, "->", build_valid(s, lk))

))())) 010100 -> ()()()
()() 0000 -> (())
) 0 -> None
(((())(((()) 111111010111 -> (((()))()())


**Độ phức tạp cách 4:** O(n) thời gian, O(n) bộ nhớ cho hai ngăn xếp.

## 8. TRACE từng bước

### 8.1 Trace cách 2 (`[lo, hi]`) trên ví dụ 4: `s = "(((())(((())"`, `locked = "111111010111"`

Hai vị trí tự do là chỉ số 6 và 8. Bảng kỳ vọng (sau mỗi ký tự):

| i | s[i] | khóa | lo | hi |
|---|---|---|---|---|
| 0..3 | ( ( ( ( | 1 | 4 | 4 |
| 4 | ) | 1 | 3 | 3 |
| 5 | ) | 1 | 2 | 2 |
| 6 | ? | 0 | 1 | 3 |
| 7 | ( | 1 | 2 | 4 |
| 8 | ? | 0 | 1 | 5 |
| 9 | ( | 1 | 2 | 6 |
| 10 | ) | 1 | 1 | 5 |
| 11 | ) | 1 | 0 | 4 |

Kết thúc `lo = 0` ⇒ `True`.

In [16]:
def trace_range(s, locked):
    lo = hi = 0
    print(" i  s  k   lo  hi")
    for i, (c, k) in enumerate(zip(s, locked)):
        if k == "0":
            lo, hi = lo - 1, hi + 1
        elif c == "(":
            lo, hi = lo + 1, hi + 1
        else:
            lo, hi = lo - 1, hi - 1
        note = ""
        if hi < 0:
            print(f"{i:>2}  {c}  {k}  {lo:>3} {hi:>3}   hi<0 -> False"); return False
        if lo < 0:
            lo = max(lo, 0); note = "  (kẹp lo về 0)"
        print(f"{i:>2}  {c}  {k}  {lo:>3} {hi:>3}{note}")
    print("kết quả:", len(s) % 2 == 0 and lo == 0)

trace_range("(((())(((())", "111111010111")

 i  s  k   lo  hi
 0  (  1    1   1
 1  (  1    2   2
 2  (  1    3   3
 3  (  1    4   4
 4  )  1    3   3
 5  )  1    2   2
 6  (  0    1   3
 7  (  1    2   4
 8  (  0    1   5
 9  (  1    2   6
10  )  1    1   5
11  )  1    0   4
kết quả: True


### 8.2 Trace cách 3 (hai chiều) trên ví dụ thất bại `s = "(()("`, `locked = "1101"`

Trái→phải không phát hiện gì (chỉ có một `)` khóa), nhưng phải→trái phát hiện `(` khóa ở cuối không có ai bên phải để ghép.

In [17]:
def trace_two_pass(s, locked):
    x = 0
    print("Trái -> phải:")
    for i, (c, k) in enumerate(zip(s, locked)):
        x += 1 if (k == "0" or c == "(") else -1
        print(f"  i={i} '{c}' k={k} x={x}")
        if x < 0: print("  -> False"); return False
    x = 0
    print("Phải -> trái:")
    for i in range(len(s) - 1, -1, -1):
        c, k = s[i], locked[i]
        x += 1 if (k == "0" or c == ")") else -1
        print(f"  i={i} '{c}' k={k} x={x}")
        if x < 0: print("  -> False"); return False
    print("  -> True"); return True

trace_two_pass("(()(", "1101")

Trái -> phải:
  i=0 '(' k=1 x=1
  i=1 '(' k=1 x=2
  i=2 ')' k=0 x=3
  i=3 '(' k=1 x=4
Phải -> trái:
  i=3 '(' k=1 x=-1
  -> False


False

### 8.3 Trace cách 4 (hai ngăn xếp) trên ví dụ 1

In [18]:
def trace_stacks(s, locked):
    opens, free = [], []
    for i, (c, k) in enumerate(zip(s, locked)):
        if k == "0":
            free.append(i); act = "tự do -> push free"
        elif c == "(":
            opens.append(i); act = "'(' khóa -> push opens"
        elif opens:
            j = opens.pop(); act = f"')' khóa ghép với '(' khóa ở {j}"
        elif free:
            j = free.pop(); act = f"')' khóa ghép với tự do ở {j} (đổi thành '(')"
        else:
            print(f"i={i}: ')' khóa không ai ghép -> False"); return False
        print(f"i={i} '{c}' k={k}: {act:<42} opens={opens} free={free}")
    while opens and free and free[-1] > opens[-1]:
        print(f"ghép '(' khóa ở {opens[-1]} với tự do ở {free[-1]}"); opens.pop(); free.pop()
    ok = not opens and len(free) % 2 == 0
    print("còn lại opens =", opens, ", free =", free, "->", ok)
    return ok

trace_stacks("))()))", "010100")

i=0 ')' k=0: tự do -> push free                         opens=[] free=[0]
i=1 ')' k=1: ')' khóa ghép với tự do ở 0 (đổi thành '(') opens=[] free=[]
i=2 '(' k=0: tự do -> push free                         opens=[] free=[2]
i=3 ')' k=1: ')' khóa ghép với tự do ở 2 (đổi thành '(') opens=[] free=[]
i=4 ')' k=0: tự do -> push free                         opens=[] free=[4]
i=5 ')' k=0: tự do -> push free                         opens=[] free=[4, 5]
còn lại opens = [] , free = [4, 5] -> True


True

## 9. Code nộp LeetCode

Dùng cách quét hai chiều (O(n), O(1)), dễ chứng minh và khó viết sai.

In [19]:
class Solution:
    def canBeValid(self, s: str, locked: str) -> bool:
        n = len(s)
        if n % 2 == 1:
            return False
        x = 0                                  # (có thể mở) - (')' khóa)
        for i in range(n):
            if locked[i] == "0" or s[i] == "(":
                x += 1
            else:
                x -= 1
                if x < 0:
                    return False
        x = 0                                  # (có thể đóng) - ('(' khóa)
        for i in range(n - 1, -1, -1):
            if locked[i] == "0" or s[i] == ")":
                x += 1
            else:
                x -= 1
                if x < 0:
                    return False
        return True

In [20]:
sol = Solution()
for s, lk, exp in examples:
    got = sol.canBeValid(s, lk)
    print(f"{s:<14} {lk:<14} -> {got} (kỳ vọng {exp})")

))()))         010100         -> True (kỳ vọng True)
()()           0000           -> True (kỳ vọng True)
)              0              -> False (kỳ vọng False)
(((())(((())   111111010111   -> True (kỳ vọng True)


## 10. Vì sao thuật toán đúng

**Điều kiện cần.** Giả sử có cách gán hợp lệ `t`.
- `n` phải chẵn (mục 3.4).
- Xét tiền tố bất kỳ `t[0..i]`: balance ≥ 0 ⇒ số `(` ≥ số `)` trong tiền tố. Mọi `(` trong `t` nằm ở vị trí `(` khóa hoặc tự do; mọi `)` khóa vẫn là `)` ⇒ (số `(` khóa + số tự do) ≥ số `(` của `t` ≥ số `)` của `t` ≥ số `)` khóa. Đó chính là `x ≥ 0` ở quét trái→phải.
- Đối xứng cho hậu tố (đọc ngược chuỗi, đổi vai `(` và `)`).

**Điều kiện đủ.** Giả sử ba điều kiện thỏa. Gọi `L` = số `(` khóa, `F` = số vị trí tự do. Mọi chuỗi hợp lệ có đúng `n/2` dấu `(`, nên cần `a = n/2 − L` vị trí tự do làm `(`.
- Điều kiện phải→trái trên toàn chuỗi cho `n − L ≥ L` ⇒ `a ≥ 0`; điều kiện trái→phải trên toàn chuỗi cho `L + F ≥ n − L − F` ⇒ `a ≤ F`. Vậy `a` hợp lệ.
- Dựng `t`: cho **`a` vị trí tự do đầu tiên** (từ trái) làm `(`, còn lại làm `)`. Tổng balance = `2(L + a) − n = 0`.
- Xét tiền tố bất kỳ. *Trường hợp 1:* tiền tố chưa chứa hết `a` vị trí tự do đầu ⇒ mọi vị trí tự do trong tiền tố đều là `(` ⇒ balance = (số "có thể mở") − (số `)` khóa) ≥ 0 nhờ điều kiện trái→phải. *Trường hợp 2:* tiền tố đã chứa đủ `a` vị trí đó ⇒ mọi vị trí tự do ở hậu tố còn lại đều là `)`; vì tổng = 0 nên balance tiền tố = −(đóng góp của hậu tố) = (số "có thể đóng" ở hậu tố) − (số `(` khóa ở hậu tố) ≥ 0 nhờ điều kiện phải→trái.
- Vậy mọi tiền tố ≥ 0 và tổng = 0 ⇒ `t` hợp lệ. ∎

Trực giác: đặt `(` càng sớm càng tốt làm **mọi** balance tiền tố đồng thời lớn nhất; nếu ngay cả cách đó cũng hỏng thì không cách nào thành công.

Trong notebook, ta không chỉ tin lập luận: mục 12 kiểm tra `build_valid` tạo ra chuỗi **thực sự hợp lệ** và **giữ nguyên mọi vị trí khóa** mỗi khi đáp án là `True`, đối chiếu với vét cạn.

**Cách 2 đúng** vì `[lo, hi]` (kèm chẵn lẻ) mô tả chính xác tập balance khả dĩ ở mục 4 — chứng minh bằng quy nạp: từ một đoạn cùng chẵn lẻ, các phép ±1 cho ra đoạn cùng chẵn lẻ; bỏ phần âm vẫn là đoạn. `hi` là balance khi mọi `?` thành `(` — nếu nó âm thì mọi cách đều âm.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `k` = số vị trí tự do.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | O(2^k · n) | O(n) |
| 1b. DP tập balance | O(n²) | O(n) |
| 2. `[lo, hi]` | O(n) | O(1) |
| 3. Hai chiều | O(n) (2 lượt) | O(1) |
| 4. Hai ngăn xếp | O(n) | O(n) |

Đo thời gian trên `n = 10⁵`:

In [21]:
rng = random.Random(1)
n = 10**5
s_big = "".join(rng.choice("()") for _ in range(n))
lk_big = "".join(rng.choice("01") for _ in range(n))
for name, f in [("range", range_lo_hi), ("two_pass", two_pass), ("stacks", stacks),
                ("Solution", Solution().canBeValid)]:
    t = time.perf_counter(); r = f(s_big, lk_big); dt = time.perf_counter() - t
    print(f"{name:<9} -> {r}  {dt*1000:.1f} ms")

range     -> True  7.0 ms
two_pass  -> True  11.9 ms


stacks    -> True  7.1 ms
Solution  -> True  10.0 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [22]:
ALL = {
    "brute": brute, "dp_set": dp_set, "range": range_lo_hi,
    "range_LC": SolutionRange().canBeValid, "two_pass": two_pass,
    "two_pass_LC": SolutionTwoPass().canBeValid, "stacks": stacks,
    "build": lambda s, l: build_valid(s, l) is not None,
    "Solution": Solution().canBeValid,
}
edge = examples + [
    ("()", "11", True), (")(", "11", False), (")(", "00", True), ("((", "10", True),
    ("((", "01", False), ("))", "10", False), ("))", "01", True),
    ("(()(", "1101", False),        # '(' khóa cuối không có ai đóng
    ("))((", "0110", True),         # đổi đầu/cuối: ()()
    (")))(((", "000000", True), (")))(((", "111000", False),
    ("((((((", "000111", False),   # 3 '(' khóa cuối cần 3 ')' bên phải: không có
    ("((((((", "111000", True),
]
for s, lk, exp in edge:
    for name, f in ALL.items():
        assert f(s, lk) == exp, (name, s, lk, exp)
print("Edge case OK:", len(edge), "case x", len(ALL), "cách")

Edge case OK: 17 case x 9 cách


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (n ≤ 12) + kiểm tra chuỗi dựng được

In [23]:
rng = random.Random(2116)
true_cnt = 0
for _ in range(6000):
    n = rng.randint(1, 12)
    s = "".join(rng.choice("()") for _ in range(n))
    lk = "".join(rng.choice("01") for _ in range(n))
    ref = brute(s, lk)
    true_cnt += ref
    for name, f in ALL.items():
        assert f(s, lk) == ref, (name, s, lk, ref)
    t = build_valid(s, lk)
    if ref:
        assert is_valid(t) and all(t[i] == s[i] for i in range(n) if lk[i] == "1"), (s, lk, t)
print("6000 case ngẫu nhiên OK, số case True:", true_cnt)

6000 case ngẫu nhiên OK, số case True: 1396


### 12.3 Đối chiếu các cách O(n) với DP tập balance trên input vừa (n ≤ 300)

In [24]:
rng = random.Random(99)
for _ in range(400):
    n = rng.choice([2, 4, 50, 120, 300])
    p = rng.random()
    s = "".join("(" if rng.random() < p else ")" for _ in range(n))
    lk = "".join("1" if rng.random() < rng.random() else "0" for _ in range(n))
    ref = dp_set(s, lk)
    for name in ["range", "range_LC", "two_pass", "stacks", "build", "Solution"]:
        assert ALL[name](s, lk) == ref, (name, s, lk)
print("400 case vừa OK")

400 case vừa OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra độ dài lẻ (với cách `[lo, hi]`)

In [25]:
def wrong_no_parity(s, locked):
    lo = hi = 0
    for c, k in zip(s, locked):
        if k == "0": lo, hi = lo - 1, hi + 1
        elif c == "(": lo, hi = lo + 1, hi + 1
        else: lo, hi = lo - 1, hi - 1
        if hi < 0: return False
        lo = max(lo, 0)
    return lo == 0

print("bản sai :", wrong_no_parity(")", "0"))
print("bản đúng:", Solution().canBeValid(")", "0"))

bản sai : True
bản đúng: False


**Vì sao sai:** `max(lo, 0)` làm `lo` lệch chẵn lẻ (−1 thành 0 thay vì 1). Lập luận "lo ghi = 0 ⇒ lo thật = 0" chỉ đúng khi `n` chẵn. **Sửa:** thêm `if len(s) % 2: return False`, hoặc kẹp chuẩn chẵn lẻ (`lo = 1` khi `lo = −1`).

### Lỗi 2 — Chỉ quét một chiều (trái→phải)

In [26]:
def wrong_one_direction(s, locked):
    if len(s) % 2: return False
    x = 0
    for c, k in zip(s, locked):
        x += 1 if (k == "0" or c == "(") else -1
        if x < 0: return False
    return True

for s, lk in [("((", "11"), ("(()(", "1101")]:
    print(s, lk, "bản sai:", wrong_one_direction(s, lk), "| bản đúng:", Solution().canBeValid(s, lk))

(( 11 bản sai: True | bản đúng: False
(()( 1101 bản sai: True | bản đúng: False


**Vì sao sai:** quét trái→phải chỉ bảo đảm mỗi `)` khóa có "bạn" bên trái; nó không kiểm tra mỗi `(` khóa có bạn bên phải. **Sửa:** thêm lượt phải→trái (hoặc dùng `[lo, hi]` với điều kiện cuối `lo == 0`).

### Lỗi 3 — Tin vào giá trị ban đầu của ký tự tự do

In [27]:
def wrong_trust_free(s, locked):
    if len(s) % 2: return False
    x = 0
    for c, k in zip(s, locked):
        x += 1 if c == "(" else (-1 if k == "1" else 0)   # SAI: ')' tự do bị coi là "không mở được"
        if x < 0: return False
    x = 0
    for c, k in zip(reversed(s), reversed(locked)):
        x += 1 if c == ")" else (-1 if k == "1" else 0)
        if x < 0: return False
    return True

print("bản sai :", wrong_trust_free("))()))", "010100"))
print("bản đúng:", Solution().canBeValid("))()))", "010100"))

bản sai : False
bản đúng: True


**Vì sao sai:** ở ví dụ 1, `s[0] = ')'` nhưng tự do — nó *có thể* thành `(`. Bản sai không đếm nó là ứng viên mở. **Sửa:** điều kiện "có thể mở" là `locked[i] == '0' or s[i] == '('`, không phụ thuộc `s[i]` khi tự do.

### Lỗi 4 — Chỉ đếm tổng (không xét thứ tự)

Đếm "số `(` khóa ≤ n/2 và số `)` khóa ≤ n/2" là điều kiện cần nhưng không đủ: vị trí mới là vấn đề.

In [28]:
def wrong_count_only(s, locked):
    n = len(s)
    if n % 2: return False
    lo_cnt = sum(1 for c, k in zip(s, locked) if k == "1" and c == "(")
    lc_cnt = sum(1 for c, k in zip(s, locked) if k == "1" and c == ")")
    return lo_cnt <= n // 2 and lc_cnt <= n // 2

s, lk = ")(", "11"
print("bản sai :", wrong_count_only(s, lk), "| bản đúng:", Solution().canBeValid(s, lk))
s, lk = "((((((", "000111"
print("bản sai :", wrong_count_only(s, lk), "| bản đúng:", Solution().canBeValid(s, lk))

bản sai : True | bản đúng: False
bản sai : True | bản đúng: False


**Sửa:** điều kiện phải được kiểm tra trên **mọi tiền tố/hậu tố**, không chỉ trên toàn chuỗi.

### Lỗi 5 — Cách ngăn xếp: ghép `(` khóa còn dư với vị trí tự do *bất kỳ* (kể cả bên trái)

In [29]:
def wrong_stacks_any_side(s, locked):
    opens, free = [], []
    for i, (c, k) in enumerate(zip(s, locked)):
        if k == "0": free.append(i)
        elif c == "(": opens.append(i)
        elif opens: opens.pop()
        elif free: free.pop()
        else: return False
    # SAI: chỉ so số lượng, không so vị trí
    return len(free) >= len(opens) and (len(free) - len(opens)) % 2 == 0

s, lk = "((", "01"
print("bản sai :", wrong_stacks_any_side(s, lk), "| bản đúng:", Solution().canBeValid(s, lk))

bản sai : True | bản đúng: False


**Vì sao sai:** `(` khóa ở chỉ số 1 cần một `)` ở **bên phải**, nhưng vị trí tự do duy nhất (chỉ số 0) ở bên trái. **Sửa:** ghép từ đỉnh hai ngăn xếp với điều kiện `free[-1] > opens[-1]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài ngoặc + "ký tự đổi được / joker / `*`" ⇒ nghĩ ngay tới **khoảng balance `[lo, hi]`** hoặc **quét hai chiều**. Bài anh em: 678 Valid Parenthesis String (`*` có thể là rỗng — khi đó không có ràng buộc chẵn lẻ).
- Luôn kiểm tra **n lẻ ⇒ False** trước tiên — một dòng tiết kiệm rất nhiều lỗi.
- Ký tự tự do: **quên giá trị ban đầu** của nó; chỉ khóa mới quan trọng.
- Khi cần "dựng" lời giải cụ thể (không chỉ True/False) ⇒ dùng **ngăn xếp chỉ số**.
- Khi nghi ngờ tham lam, viết vét cạn 2^k cho n ≤ 12 và đối chiếu ngẫu nhiên — mất 2 phút, tiết kiệm nhiều lần WA.
- Bài cùng họ: 20 Valid Parentheses, 921 Minimum Add to Make Parentheses Valid, 1249 Minimum Remove to Make Valid Parentheses, 1541 Minimum Insertions to Balance a Parentheses String.

## 15. Biến thể của bài

### 15.1 Biến thể A — Dựng **một** chuỗi hợp lệ (hoặc trả `None`)

Đã cài ở `build_valid` (mục 7.3). Ở đây thêm kiểm tra: chuỗi dựng ra giữ nguyên vị trí khóa và hợp lệ trên các ví dụ.

In [30]:
for s, lk, exp in edge:
    t = build_valid(s, lk)
    if exp:
        assert is_valid(t) and all(t[i] == s[i] for i in range(len(s)) if lk[i] == "1")
    else:
        assert t is None
    print(f"{s:<14}{lk:<14} -> {t}")

))()))        010100         -> ()()()
()()          0000           -> (())
)             0              -> None
(((())(((())  111111010111   -> (((()))()())
()            11             -> ()
)(            11             -> None
)(            00             -> ()
((            10             -> ()
((            01             -> None
))            10             -> None
))            01             -> ()
(()(          1101           -> None
))((          0110           -> ()()
)))(((        000000         -> ((()))
)))(((        111000         -> None
((((((        000111         -> None
((((((        111000         -> ((()))


### 15.2 Biến thể B — **Số lần đổi tối thiểu**

Đổi một vị trí tự do = gán nó khác giá trị ban đầu. Tìm số lần đổi ít nhất để `s` hợp lệ (hoặc −1). DP trên balance: `dp[b]` = số lần đổi ít nhất để đạt balance `b` sau tiền tố hiện tại. O(n²) — đủ cho n vài nghìn.

In [31]:
def min_changes(s, locked):
    n = len(s)
    if n % 2: return -1
    INF = float("inf")
    dp = {0: 0}
    for c, k in zip(s, locked):
        nd = {}
        for b, cost in dp.items():
            opts = [c] if k == "1" else ["(", ")"]
            for o in opts:
                nb = b + (1 if o == "(" else -1)
                if nb < 0 or nb > n: continue
                nc = cost + (1 if o != c else 0)
                if nc < nd.get(nb, INF): nd[nb] = nc
        dp = nd
    return dp.get(0, -1)

def min_changes_brute(s, locked):
    free = [i for i, k in enumerate(locked) if k == "0"]
    best = -1
    for choice in itertools.product("()", repeat=len(free)):
        t = list(s)
        for i, c in zip(free, choice): t[i] = c
        if is_valid(t):
            cost = sum(1 for i in free if t[i] != s[i])
            if best == -1 or cost < best: best = cost
    return best

print(min_changes("))()))", "010100"), min_changes("()()", "0000"), min_changes(")", "0"))
rng = random.Random(5)
for _ in range(1500):
    n = rng.randint(1, 10)
    s = "".join(rng.choice("()") for _ in range(n)); lk = "".join(rng.choice("01") for _ in range(n))
    assert min_changes(s, lk) == min_changes_brute(s, lk)
print("min_changes khớp vét cạn")

2 0 -1
min_changes khớp vét cạn


### 15.3 Biến thể C — Ký tự tự do có thể **xóa** (giống LC 678)

Nếu `?` có thể là `(`, `)` hoặc **rỗng**, ràng buộc chẵn lẻ biến mất: cập nhật `?` thành `lo -= 1, hi += 1` nhưng tập balance không còn "cùng chẵn lẻ" mà là **mọi số** trong `[lo, hi]`. Khi đó `max(lo, 0)` là chính xác tuyệt đối và không cần kiểm tra `n` chẵn.

In [32]:
def can_be_valid_with_empty(s, locked):
    lo = hi = 0
    for c, k in zip(s, locked):
        if k == "0": lo, hi = lo - 1, hi + 1
        elif c == "(": lo, hi = lo + 1, hi + 1
        else: lo, hi = lo - 1, hi - 1
        if hi < 0: return False
        lo = max(lo, 0)
    return lo == 0

print(can_be_valid_with_empty(")", "0"))      # '?' thành rỗng -> chuỗi rỗng; coi là hợp lệ ở biến thể này
print(can_be_valid_with_empty("(()", "100"))  # s[1] là '?' -> rỗng: "()" -> True
print(can_be_valid_with_empty("(()", "110"))  # chỉ s[2] tự do: "((("/"(()"/"((" đều sai -> False

True
True
False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử 2^k cách gán | O(2^k·n) | O(n) | kiểm chứng, n rất nhỏ |
| 1b. DP tập balance | lưu mọi balance khả dĩ | O(n²) | O(n) | dễ nghĩ, mở rộng sang "min đổi" |
| 2. `[lo, hi]` | tập balance là một đoạn cùng chẵn lẻ | O(n) | O(1) | gọn nhất, nhớ kèm kiểm tra n chẵn |
| 3. Quét hai chiều | tiền tố đủ "mở", hậu tố đủ "đóng" | O(n) | O(1) | **mặc định**, dễ chứng minh |
| 4. Hai ngăn xếp | ghép cụ thể từng cặp | O(n) | O(n) | khi cần **dựng** chuỗi kết quả |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `canBeValid("))((", "0000")`, `canBeValid("))((", "1001")`, `canBeValid("(((", "000")` lần lượt là gì?

**Bài 2 (tính tay):** Tính `[lo, hi]` sau từng ký tự cho `s = "())(()"`, `locked = "101010"`. Kết quả cuối là gì?

**Bài 3 (điền TODO):** Viết `bt3_todo(s, locked)` trả về **số vị trí tự do** cần gán thành `(` trong *mọi* chuỗi hợp lệ (gợi ý: tổng số `(` phải là n/2). Trả −1 nếu không thể hợp lệ.

**Bài 4 (tự viết lại):** Viết `bt4_todo(s, locked)` cài lại cách `[lo, hi]` **bản chuẩn chẵn lẻ** (khi `lo` âm thì đặt `lo = 1`), *không* kiểm tra `n` chẵn ở đầu mà chỉ dựa vào điều kiện cuối.

**Bài 5 (mini challenge):** Viết `bt5_todo(s, locked)` trả về **số cách gán khác nhau** cho các vị trí tự do để `s` hợp lệ, modulo 10⁹+7 (DP trên balance, n ≤ 2000).

In [33]:
def bt3_todo(s, locked):
    # TODO: số vị trí tự do phải là '(' (hoặc -1)
    pass

def bt4_todo(s, locked):
    # TODO: [lo, hi] chuẩn chẵn lẻ, không kiểm tra n chẵn ở đầu
    pass

def bt5_todo(s, locked):
    # TODO: đếm số cách gán hợp lệ mod 1e9+7
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: n chẵn chưa? Mức 2: với `"1001"` thì `)` khóa ở đầu. Mức 3: `True` (thành `()()`), `False` (`)` khóa đầu không có ai bên trái), `False` (n lẻ).

**Bài 2.** Mức 1: vị trí tự do là chỉ số 1, 3, 5. Mức 2: bắt đầu `(` khóa ⇒ [1,1]. Mức 3: dãy `[lo, hi]` là [1,1] → [0,2] → [0,1] (kẹp từ −1) → [0,2] (kẹp) → [1,3] → [0,4]; cuối `lo = 0`.

**Bài 3.** Mức 1: mọi chuỗi hợp lệ có đúng n/2 dấu `(`. Mức 2: số `(` khóa cố định. Mức 3: đáp án `n/2 − (số '(' khóa)` nếu `canBeValid`, ngược lại −1.

**Bài 4.** Mức 1: giữ bất biến lo, hi cùng chẵn lẻ với i. Mức 2: khi `lo = −1` thì giá trị nhỏ nhất không âm cùng chẵn lẻ là 1. Mức 3: cuối cùng trả `lo == 0` — nếu n lẻ thì `lo` lẻ, tự động `False`.

**Bài 5.** Mức 1: `dp[b]` = số cách đạt balance `b`. Mức 2: khóa ⇒ một chuyển trạng thái; tự do ⇒ hai. Mức 3: bỏ `b < 0`, đáp án `dp[0]` sau n ký tự; dùng list độ dài n+2 để nhanh.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `True`, `False`, `False`.

**Bài 2:** xem output của `trace_range` dưới đây; kết quả `True` (ví dụ `"()()()"`).

In [34]:
trace_range("())(()", "101010")

def bt3(s, locked):
    if not Solution().canBeValid(s, locked):
        return -1
    return len(s) // 2 - sum(1 for c, k in zip(s, locked) if k == "1" and c == "(")

def bt4(s, locked):
    lo = hi = 0
    for c, k in zip(s, locked):
        if k == "0": lo, hi = lo - 1, hi + 1
        elif c == "(": lo, hi = lo + 1, hi + 1
        else: lo, hi = lo - 1, hi - 1
        if hi < 0: return False
        if lo < 0: lo = 1
    return lo == 0

def bt5(s, locked, MOD=10**9 + 7):
    n = len(s)
    dp = [0] * (n + 2); dp[0] = 1
    for c, k in zip(s, locked):
        nd = [0] * (n + 2)
        for b in range(n + 1):
            v = dp[b]
            if not v: continue
            if k == "0" or c == "(":
                nd[b + 1] = (nd[b + 1] + v) % MOD
            if (k == "0" or c == ")") and b > 0:
                nd[b - 1] = (nd[b - 1] + v) % MOD
        dp = nd
    return dp[0]

def bt5_brute(s, locked):
    free = [i for i, k in enumerate(locked) if k == "0"]; cnt = 0
    for ch in itertools.product("()", repeat=len(free)):
        t = list(s)
        for i, c in zip(free, ch): t[i] = c
        cnt += is_valid(t)
    return cnt

assert [Solution().canBeValid(*a) for a in [("))((", "0000"), ("))((", "1001"), ("(((", "000")]] == [True, False, False]
assert Solution().canBeValid("())(()", "101010") is True
assert bt3("))()))", "010100") == 3 and bt3(")", "0") == -1 and bt3("((((((", "111000") == 0
assert bt5("((((", "0000") == 2         # (()) và ()()
rng = random.Random(1919)
for _ in range(2000):
    n = rng.randint(1, 11)
    s = "".join(rng.choice("()") for _ in range(n)); lk = "".join(rng.choice("01") for _ in range(n))
    ref = brute(s, lk)
    assert bt4(s, lk) == ref
    assert (bt3(s, lk) == -1) == (not ref)
    assert bt5(s, lk) == bt5_brute(s, lk)
    if ref:
        t = build_valid(s, lk)
        assert sum(1 for i in range(n) if lk[i] == "0" and t[i] == "(") == bt3(s, lk)
print("Đáp án bài tập: tất cả assert OK")

 i  s  k   lo  hi
 0  (  1    1   1
 1  )  0    0   2
 2  )  1    0   1  (kẹp lo về 0)
 3  (  0    0   2  (kẹp lo về 0)
 4  (  1    1   3
 5  )  0    0   4
kết quả: True


Đáp án bài tập: tất cả assert OK


Ghi chú bài 3: một hệ quả thú vị — mọi chuỗi hợp lệ dựng được đều dùng **cùng số** `(` ở vị trí tự do; khác nhau chỉ ở *vị trí* đặt chúng.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def canBeValid(s, locked):
    if len(s) % 2: return False
    x = 0
    for c, k in zip(s, locked):                       # trái -> phải
        x += 1 if (k == '0' or c == '(') else -1
        if x < 0: return False
    x = 0
    for c, k in zip(reversed(s), reversed(locked)):   # phải -> trái
        x += 1 if (k == '0' or c == ')') else -1
        if x < 0: return False
    return True
```

**Checklist:**
- [ ] `n` lẻ ⇒ `False` ngay.
- [ ] Vị trí tự do: **không** nhìn `s[i]`; nó là ứng viên cho cả hai vai.
- [ ] Đủ **hai chiều** (hoặc `[lo, hi]` với `lo == 0` ở cuối).
- [ ] Nếu dùng `[lo, hi]`: kiểm tra `hi < 0` **trước** khi kẹp `lo`.
- [ ] Đã thử: `")"/"0"`, `"(("/"01"`, `")("/"11"`, ví dụ 1 (ký tự `)` tự do ở đầu).